# Multi-Tool AI Support Agent

A model-agnostic implementation of a single AI agent with multiple tools.

Detailed architecture, integration instructions, API contracts, frontend guidance,
and component explanations are available in `AGENT_EXPLANATION.md`.

## 0. Setup

Run the notebook in order. MOCK mode is enabled by default so the complete pipeline
can be tested without external API keys or a specific language model.

In [ ]:
# Install only the libraries used by the optional live tools.
# The core routing/orchestration logic itself is model-independent.

!pip -q install -U huggingface_hub tavily-python pillow

In [ ]:
# Core imports shared by all modules.

import os
import re
import json
from pathlib import Path
from getpass import getpass
from typing import Dict, Any, Callable

from PIL import Image, ImageDraw

# Generated images and temporary outputs are stored here.
OUTPUT_DIR = Path('/content/multi_tool_agent_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Keep both external tools in mock mode until integration is verified.
VISUAL_MODE = 'mock'
SEARCH_MODE = 'mock'

print('Output directory:', OUTPUT_DIR)
print('Visual mode:', VISUAL_MODE)
print('Search mode:', SEARCH_MODE)

## 1. Mandatory Safety Guard

Runs before model-driven tool execution.

In [ ]:
class SafetyGuard:
    """
    Mandatory pre-routing safety layer.

    Stable output contract:
    {
        "level": "routine" | "review" | "urgent",
        "allow_tools": bool,
        "message": str
    }

    Integration note:
    - This does not diagnose a condition.
    - It only decides whether normal automated tools may continue.
    - A production system should use a stronger, reviewed safety layer.
    """

    URGENT = [
        # Arabic examples.
        r"أريد\s+أن\s+أقتل\s+نفسي",
        r"اريد\s+ان\s+اقتل\s+نفسي",
        r"نبي\s+نقتل\s+روحي",
        r"بنقتل\s+روحي",
        r"سأنتحر",
        r"سوف\s+انتحر",
        r"انتحار",
        # English examples.
        r"suicid",
        r"kill\s+myself",
        r"hurt\s+myself",
    ]

    MEDICAL = [
        r"تشخيص",
        r"شخ[صّ]ص",
        r"دواء",
        r"جرعة",
        r"medication",
        r"diagnos",
        r"dosage",
    ]

    def check(self, text: str) -> Dict[str, Any]:
        # Normalize spacing and case before matching.
        normalized = ' '.join(text.lower().strip().split())

        if any(re.search(pattern, normalized, re.I) for pattern in self.URGENT):
            return {
                'level': 'urgent',
                'allow_tools': False,
                'message': (
                    'The message may indicate an urgent safety concern. '
                    'Normal automation should stop and the application should switch '
                    'to the project\'s human/emergency-support flow.'
                )
            }

        if any(re.search(pattern, normalized, re.I) for pattern in self.MEDICAL):
            return {
                'level': 'review',
                'allow_tools': False,
                'message': (
                    'This system must not diagnose, prescribe medication, or recommend dosage. '
                    'It may provide general support or help the user reach a qualified professional.'
                )
            }

        return {
            'level': 'routine',
            'allow_tools': True,
            'message': 'Normal non-diagnostic support flow is allowed.'
        }


safety_guard = SafetyGuard()

# Standalone test for this module.
print(safety_guard.check('I am stressed about my exam and want a calming exercise.'))

## 2. Visual Exercise Tool

Creates visual aids only for exercises defined in the approved allow-list.

In [ ]:
# Approved exercise library.
# I keep the Arabic text because this project targets Arabic-speaking users,
# while all implementation comments and integration documentation are in English.

APPROVED_EXERCISES = {
    'box_breathing': {
        'title_ar': 'التنفس المربع',
        'title_en': 'Box Breathing',
        'steps_ar': [
            'شهيق هادئ 4 ثوانٍ.',
            'احتفاظ مريح 4 ثوانٍ.',
            'زفير هادئ 4 ثوانٍ.',
            'انتظار 4 ثوانٍ ثم التكرار عند الراحة.'
        ],
        'visual_description': (
            'four clear panels: inhale, pause, exhale, pause; '
            'simple neutral human figure'
        )
    },
    'grounding_54321': {
        'title_ar': 'تمرين 5-4-3-2-1 للتركيز على الحاضر',
        'title_en': '5-4-3-2-1 Grounding',
        'steps_ar': [
            'لاحظ 5 أشياء تستطيع رؤيتها.',
            'لاحظ 4 أشياء تستطيع لمسها.',
            'لاحظ 3 أصوات تستطيع سماعها.',
            'لاحظ رائحتين تستطيع تمييزهما.',
            'لاحظ شيئًا واحدًا تستطيع تذوقه أو تتخيله.'
        ],
        'visual_description': (
            'five-part sensory grounding illustration for sight, touch, hearing, smell and taste'
        )
    }
}

In [ ]:
def generate_exercise_visual(exercise_name: str, mode='mock') -> Dict[str, Any]:
    """
    Generate a visual aid for one approved exercise.

    The function is provider-independent from the orchestrator's point of view.
    Only this function needs to change if we switch image-generation APIs.
    """

    # Block any exercise that is not explicitly approved.
    if exercise_name not in APPROVED_EXERCISES:
        return {
            'status': 'blocked',
            'reason': 'Exercise not approved',
            'approved': list(APPROVED_EXERCISES)
        }

    exercise = APPROVED_EXERCISES[exercise_name]

    # I intentionally ask the image model not to render text.
    # Arabic labels should be rendered by the frontend for better accuracy.
    prompt = (
        'Create a clean, calm, non-medical educational illustration for a youth wellbeing app. '
        f"Show {exercise['visual_description']}. "
        'Simple modern flat or soft 3D style, neutral person, modest clothing, '
        'white background, clear sequence, no diagnosis, no medication, no logos, no embedded text.'
    )

    if mode == 'mock':
        # Local placeholder: this verifies the pipeline without any external image API.
        image = Image.new('RGB', (1000, 650), 'white')
        draw = ImageDraw.Draw(image)
        draw.rectangle((40, 40, 960, 610), outline='black', width=3)
        draw.text((70, 80), f'MOCK VISUAL: {exercise_name}', fill='black')
        draw.text((70, 130), 'Pipeline test only. Switch to live mode for an image API.', fill='black')

        path = OUTPUT_DIR / f'{exercise_name}_mock.png'
        image.save(path)

        return {
            'status': 'ok',
            'mode': 'mock',
            'exercise_name': exercise_name,
            'title_ar': exercise['title_ar'],
            'title_en': exercise['title_en'],
            'steps_ar': exercise['steps_ar'],
            'image_path': str(path),
            'prompt': prompt
        }

    # Example live provider: Hugging Face Inference Providers.
    # This block may be replaced with any image-generation API.
    from huggingface_hub import InferenceClient

    token = os.environ.get('HF_TOKEN')
    if not token:
        raise ValueError('HF_TOKEN is missing')

    model_id = os.environ.get('IMAGE_MODEL', 'black-forest-labs/FLUX.1-schnell')
    client = InferenceClient(api_key=token, provider='auto')
    image = client.text_to_image(
        prompt=prompt,
        model=model_id,
        width=1024,
        height=1024
    )

    path = OUTPUT_DIR / f'{exercise_name}_live.png'
    image.save(path)

    return {
        'status': 'ok',
        'mode': 'live',
        'exercise_name': exercise_name,
        'title_ar': exercise['title_ar'],
        'title_en': exercise['title_en'],
        'steps_ar': exercise['steps_ar'],
        'image_path': str(path),
        'prompt': prompt
    }


# Standalone test.
visual_test = generate_exercise_visual('box_breathing', 'mock')
print(json.dumps(visual_test, ensure_ascii=False, indent=2))
display(Image.open(visual_test['image_path']))

## 3. Professional Support Search Tool

Returns public-source candidates based on support type and location.

In [ ]:
# Categories the model/router is allowed to request.
ALLOWED_SUPPORT_TYPES = {
    'psychologist': ['psychologist', 'clinical psychologist', 'أخصائي نفسي'],
    'psychiatrist': ['psychiatrist', 'طبيب نفسي'],
    'counselor': ['counselor', 'counselling', 'مرشد نفسي'],
    'mental_health_clinic': ['mental health clinic', 'عيادة نفسية', 'الصحة النفسية']
}


def find_professional_support(
    support_type: str,
    city: str,
    country='Libya',
    mode='mock'
) -> Dict[str, Any]:
    """
    Search for public professional-support sources.

    Only this function needs to change if Tavily is replaced with
    Google Places, a local verified directory, or another search provider.
    """

    if support_type not in ALLOWED_SUPPORT_TYPES:
        return {
            'status': 'blocked',
            'allowed': list(ALLOWED_SUPPORT_TYPES)
        }

    city = (city or '').strip()

    if not city:
        return {
            'status': 'needs_input',
            'missing': ['city'],
            'message': 'Which city would you like me to search in?'
        }

    query = (
        f"{ALLOWED_SUPPORT_TYPES[support_type][0]} "
        f"{city} {country} mental health clinic contact appointment"
    )

    if mode == 'mock':
        # Deliberately fictional records for safe integration testing.
        return {
            'status': 'ok',
            'mode': 'mock',
            'query': query,
            'candidates': [
                {
                    'name_or_source': 'DEMO Provider 1',
                    'url': 'https://example.com/demo-1',
                    'verification_status': 'mock_only_not_real'
                },
                {
                    'name_or_source': 'DEMO Provider 2',
                    'url': 'https://example.com/demo-2',
                    'verification_status': 'mock_only_not_real'
                }
            ]
        }

    # Example live provider: Tavily.
    # This block can be swapped for another search/data source later.
    from tavily import TavilyClient

    key = os.environ.get('TAVILY_API_KEY')
    if not key:
        raise ValueError('TAVILY_API_KEY is missing')

    response = TavilyClient(api_key=key).search(
        query=query,
        max_results=8,
        search_depth='advanced',
        include_answer=False
    )

    candidates = []

    for item in response.get('results', []):
        candidates.append({
            'name_or_source': item.get('title', 'Public source'),
            'url': item.get('url'),
            'public_snippet': (item.get('content') or '')[:450],
            'verification_status': 'needs_human_verification'
        })

    return {
        'status': 'ok',
        'mode': 'live',
        'query': query,
        'candidates': candidates[:5]
    }


# Standalone test.
provider_test = find_professional_support('psychologist', 'Tripoli', mode='mock')
print(json.dumps(provider_test, ensure_ascii=False, indent=2))

## 4. Model-Independent Router Contract

Any language model can be connected through the shared JSON routing contract.

In [ ]:
# Shared system prompt for any selected language model.
# The adapter below passes this prompt to the selected model.

ROUTER_PROMPT = """
You are the tool-routing controller for a youth wellbeing support prototype.

You do NOT diagnose mental illness.
You do NOT prescribe medication or dosage.
A mandatory safety layer runs before you.

Your job is only to decide the next useful action.

AVAILABLE TOOLS

1) none
Use for ordinary supportive conversation when no external tool is needed.

2) visual_exercise
Use when a short, general wellbeing exercise would be useful.
Allowed exercise_name values:
- box_breathing
- grounding_54321

3) provider_search
Use when the user explicitly asks for professional support, a specialist,
or where to find professional help.
Allowed support_type values:
- psychologist
- psychiatrist
- counselor
- mental_health_clinic

If the city is not stated, set city to null so the application can ask for it.
Never call a provider the "best doctor" unless the application has a verified,
objective ranking source.

Return ONLY valid JSON in exactly this structure:
{
  "response": "short supportive response in the same language as the user",
  "tool": "none | visual_exercise | provider_search",
  "tool_input": {
    "exercise_name": null,
    "support_type": null,
    "city": null,
    "country": "Libya"
  }
}
"""

In [ ]:
def extract_json_object(text: str) -> Dict[str, Any]:
    """Extract one JSON object from a model response."""

    text = text.strip()

    # Some models still wrap JSON in Markdown fences, so remove them if present.
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)

    start = text.find('{')
    end = text.rfind('}')

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f'No JSON object found in model output: {text}')

    return json.loads(text[start:end + 1])


def route_with_any_model(
    user_text: str,
    model_callable: Callable[[str, str], str]
) -> Dict[str, Any]:
    """
    Universal adapter used by the agent.

    A model adapter only needs to provide a function with this signature:

        model_callable(system_prompt, user_text) -> raw_text

    The selected model may be local, cloud-based, or behind our own backend.
    """

    raw_output = model_callable(ROUTER_PROMPT, user_text)
    decision = extract_json_object(raw_output)

    # Minimal validation before the orchestrator trusts the model decision.
    allowed_tools = {'none', 'visual_exercise', 'provider_search'}

    if decision.get('tool') not in allowed_tools:
        raise ValueError(f"Invalid tool returned by model: {decision.get('tool')}")

    if not isinstance(decision.get('tool_input'), dict):
        raise ValueError('tool_input must be a JSON object')

    return decision


def mock_model_callable(system_prompt: str, user_text: str) -> str:
    """
    Development-only fake model.

    This mock adapter allows frontend and backend integration testing immediately,
    even before the final LLM is connected.
    """

    text = user_text.lower()

    if any(word in text for word in ['specialist', 'psychologist', 'doctor', 'مختص', 'أخصائي', 'طبيب']):
        city = 'Tripoli' if ('tripoli' in text or 'طرابلس' in text) else None
        return json.dumps({
            'response': 'I can help you look for public professional-support options.',
            'tool': 'provider_search',
            'tool_input': {
                'exercise_name': None,
                'support_type': 'psychologist',
                'city': city,
                'country': 'Libya'
            }
        }, ensure_ascii=False)

    if any(word in text for word in ['exercise', 'breathing', 'stress', 'تمرين', 'تنفس', 'متوتر']):
        return json.dumps({
            'response': 'A short breathing exercise may be useful here.',
            'tool': 'visual_exercise',
            'tool_input': {
                'exercise_name': 'box_breathing',
                'support_type': None,
                'city': None,
                'country': 'Libya'
            }
        }, ensure_ascii=False)

    return json.dumps({
        'response': 'I can continue with normal supportive conversation.',
        'tool': 'none',
        'tool_input': {
            'exercise_name': None,
            'support_type': None,
            'city': None,
            'country': 'Libya'
        }
    }, ensure_ascii=False)


# Router test without any real LLM.
router_test = route_with_any_model(
    'I am stressed and want a breathing exercise.',
    mock_model_callable
)
print(json.dumps(router_test, ensure_ascii=False, indent=2))

## 5. Multi-Tool Agent Orchestrator

Combines safety checking, model routing, tool execution, and structured output.

In [ ]:
class MultiToolSupportAgent:
    """
    One model-independent multi-tool agent.

    The selected model is injected through model_callable.
    This keeps the orchestrator independent from a specific model SDK.
    """

    def __init__(
        self,
        model_callable: Callable[[str, str], str],
        visual_mode='mock',
        search_mode='mock'
    ):
        self.model_callable = model_callable
        self.visual_mode = visual_mode
        self.search_mode = search_mode
        self.safety = SafetyGuard()

    def run(self, user_text: str) -> Dict[str, Any]:
        # STEP 1: Mandatory safety check before any model-driven tool execution.
        safety = self.safety.check(user_text)

        if not safety['allow_tools']:
            return {
                'status': 'safety_stop',
                'safety': safety,
                'response': safety['message'],
                'tool': 'none',
                'tool_input': {},
                'tool_result': None
            }

        # STEP 2: Ask the configured language model to choose an action.
        decision = route_with_any_model(
            user_text=user_text,
            model_callable=self.model_callable
        )

        tool = decision.get('tool', 'none')
        tool_input = decision.get('tool_input') or {}
        tool_result = None

        # STEP 3A: Visual exercise path.
        if tool == 'visual_exercise':
            tool_result = generate_exercise_visual(
                tool_input.get('exercise_name'),
                self.visual_mode
            )

        # STEP 3B: Professional-support search path.
        elif tool == 'provider_search':
            city = tool_input.get('city')

            if not city:
                # The frontend can use this state to ask a follow-up question.
                tool_result = {
                    'status': 'needs_input',
                    'missing': ['city'],
                    'message': 'Which city would you like me to search in?'
                }
            else:
                tool_result = find_professional_support(
                    tool_input.get('support_type') or 'psychologist',
                    city,
                    tool_input.get('country') or 'Libya',
                    self.search_mode
                )

        # STEP 4: Return one predictable object to the frontend.
        return {
            'status': 'ok',
            'safety': safety,
            'response': decision.get('response', ''),
            'tool': tool,
            'tool_input': tool_input,
            'tool_result': tool_result
        }


# Development instance.
# Replace mock_model_callable with the real model adapter when available.
agent = MultiToolSupportAgent(
    model_callable=mock_model_callable,
    visual_mode=VISUAL_MODE,
    search_mode=SEARCH_MODE
)

print('Multi-tool support agent is ready.')

## 6. End-to-End Tests

Tests the visual, provider-search, and safety paths independently.

In [ ]:
# TEST A — Visual exercise path.
result = agent.run('I am stressed about exams and want a breathing exercise.')
print(json.dumps(result, ensure_ascii=False, indent=2))

if result.get('tool') == 'visual_exercise':
    image_path = (result.get('tool_result') or {}).get('image_path')
    if image_path:
        display(Image.open(image_path))

In [ ]:
# TEST B — Professional-support search path.
result = agent.run('I want to speak with a psychologist in Tripoli.')
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# TEST C — Safety-stop path.
result = agent.run('I want to kill myself.')
print(json.dumps(result, ensure_ascii=False, indent=2))

## 7. Optional LIVE Mode

External image-generation and web-search APIs can be enabled after MOCK-mode testing succeeds.

In [ ]:
# OPTIONAL: run only when we are ready to use live external tools.

# Image-generation provider example:
# os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')
# os.environ['IMAGE_MODEL'] = 'black-forest-labs/FLUX.1-schnell'
# agent.visual_mode = 'live'

# Web-search provider example:
# os.environ['TAVILY_API_KEY'] = getpass('TAVILY_API_KEY: ')
# agent.search_mode = 'live'

print('Leave the agent in MOCK mode until the integration tests pass.')

## 8. Integration

See `AGENT_EXPLANATION.md` for model, backend, frontend, Laravel, and API integration details.